# 🌐 Marlish.AI — Cloud GPU Training Pipeline

This notebook trains the **mT5-small** model on your 18.4M parallel translation dataset, exports it to ONNX, and quantizes it for browser deployment.

### ❗ IMPORTANT: Before Running
1. Go to **Runtime → Change runtime type** → Select **T4 GPU** (NOT TPU!) → Save.
2. Upload **`ml_splits.zip`** (your `docs/Dictionary_Refs/ml_splits` folder zipped).
3. Upload **`scripts.zip`** (containing `train_model.py` and `export_onnx.py` zipped).
4. Run each cell in order from top to bottom.

In [ ]:
# ════ CELL 1: Verify GPU (MUST show 'cuda' not 'cpu') ════
import torch
import subprocess

if not torch.cuda.is_available():
    print("❌ ERROR: No GPU detected!")
    print("Go to Runtime → Change runtime type → Select T4 GPU → Save")
    print("Then restart the runtime and re-run this cell.")
    raise SystemExit("No GPU. Fix runtime type first.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU Active: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
print(f"✅ CUDA Version: {torch.version.cuda}")
print(f"✅ PyTorch: {torch.__version__}")

# Estimate training time
if 'T4' in gpu_name:
    print("\n⏱️ Estimated 500k training time: ~2-3 hours")
elif 'A100' in gpu_name or 'V100' in gpu_name:
    print("\n⏱️ Estimated 500k training time: ~30-60 minutes")
else:
    print(f"\n⏱️ GPU detected: {gpu_name} (training time will vary)")

In [ ]:
# ════ CELL 2: Install Dependencies ════
!pip install -q transformers sentencepiece datasets evaluate sacrebleu "accelerate>=1.1.0" optimum[onnxruntime] onnx onnxruntime
print("\n✅ All dependencies installed!")

In [ ]:
# ════ CELL 3: Unzip Data & Scripts ════
import os

# Check that uploads exist
for f in ['ml_splits.zip', 'scripts.zip']:
    if not os.path.exists(f):
        print(f"❌ Missing {f} — please upload it using the Files panel on the left!")
        raise SystemExit(f"Upload {f} first.")

# Unzip
!mkdir -p docs/Dictionary_Refs
!unzip -oq ml_splits.zip -d docs/Dictionary_Refs/
!mkdir -p scripts
!unzip -oq scripts.zip -d ./

# Verify files exist
for f in ['docs/Dictionary_Refs/ml_splits/train.csv', 'docs/Dictionary_Refs/ml_splits/val.csv', 'scripts/train_model.py', 'scripts/export_onnx.py']:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f"  {status} {f}")

# Show dataset size
import pandas as pd
train_rows = sum(1 for _ in open('docs/Dictionary_Refs/ml_splits/train.csv')) - 1
print(f"\n📊 Training pairs available: {train_rows:,}")

In [ ]:
# ════ CELL 4: Train the Model (500k balanced subset) ════
# To train on ALL data instead, uncomment the sed line below:
# !sed -i 's/MAX_TRAIN_SAMPLES = 500000/MAX_TRAIN_SAMPLES = None/' scripts/train_model.py

!python scripts/train_model.py

In [ ]:
# ════ CELL 5: Export to ONNX ════
import os
model_dir = 'models/marlish_mt5_finetuned'

if not os.path.exists(model_dir) or not any(f.endswith(('.bin', '.safetensors')) for f in os.listdir(model_dir)):
    print('❌ No trained model found! Make sure Cell 4 (training) completed successfully.')
    raise SystemExit('Training must complete before export.')

!python scripts/export_onnx.py

In [ ]:
# ════ CELL 6: Zip Models for Download ════
import shutil, os
from pathlib import Path

# Try quantized first, fall back to regular ONNX, then PyTorch
for name, path in [('marlish_mt5_quantized', 'models/marlish_mt5_quantized'),
                    ('marlish_mt5_onnx', 'models/marlish_mt5_onnx'),
                    ('marlish_mt5_finetuned', 'models/marlish_mt5_finetuned')]:
    if os.path.exists(path):
        size = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 1e6
        shutil.make_archive(name, 'zip', path)
        print(f'✅ Created {name}.zip ({size:.1f} MB) — download from Files panel on left!')

if not any(os.path.exists(f'{n}.zip') for n in ['marlish_mt5_quantized', 'marlish_mt5_onnx', 'marlish_mt5_finetuned']):
    print('❌ No model files found to zip. Training and export must complete first.')